# Redshift DataSet Regressor: Linear Regression

In this notebook we will build a linear regression model using the redshift dataset.  Let's dive in! 

## import libraries

In [ ]:
# arrays
import numpy as np

# unpacking files
import tarfile

# fits
from astropy.io import fits
from astropy.utils.data import download_file
from astropy.table import Table
import pandas as pd

# plotting
from matplotlib import pyplot as plt

# sklearn 
from sklearn.model_selection import train_test_split #, RandomizedSearchCV, validation_curve
#from sklearn.model_selection import KFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

## Read and Inspect Data

First we will import our data and convert it into a pandas dataaframe.

In [ ]:
file_url = 'https://archive.stsci.edu/missions/hlsp/3d-hst/RELEASE_V4.0/Photometry/3dhst_master.phot.v4.1.tar'
tarfile.open(download_file(file_url, cache=True), "r:").extract('3dhst_master.phot.v4.1/3dhst_master.phot.v4.1.cat', '.')
tab = Table.read('3dhst_master.phot.v4.1/3dhst_master.phot.v4.1.cat', format='ascii').to_pandas()
tab.head()

Next, we can start to inspect our data.

In [ ]:
tab.info()

## Filter data based on domain specific knowledge

Before building and applying a regression model, we first need to inspect and clean the dataset. This process will explore the data we have, and filter out thing like missing data or data not relevant to our task.

To explore the physical parameters of the sample, we plot the spectroscopic redshift vs. the mass derived from the FAST phototmetric fit. This is a plot to make based on astronomy specific domain knowledge.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4))

ax.scatter(tab.z_spec, tab.lmass, alpha=0.2, color='grey')
ax.set_xlim(0, 2)
ax.set_ylim(7, 12)
ax.set_xlabel(r'$z_{\rm spec}$')
ax.set_ylabel(r'$\log{(M_{*})}\,\,[M_{\odot}]$')

plt.show()

By inspection of this plot, we will only keep sources with log(M)>9 to remove sources which do not have complete coverage across redshift.

In [ ]:
print('number of datapoints originally = {}'.format(tab.shape[0]))
tab = tab[tab.lmass > 9].copy()
print('number of datapoints after drop = {}'.format(tab.shape[0]))

## Define our target variable

We're interested in predicting redshift, so our "target" variable will be z_spec. The "features" will be all columns except for the spectroscopic and photometric redshifts.

In [ ]:
target = 'z_spec'

Let's inspect our target's distribution to see how missing values are encoded.  Note: redshifts should all be positive values.

In [ ]:
fig,ax=plt.subplots()
ax.hist(tab[target]) #,  bins=1000)
ax.set_yscale('log')
#ax.set_xlim([-2,5])

Next, we will remove sources which have no constraints for our target variable:

In [ ]:
print('number of datapoints originally = {}'.format(tab.shape[0]))
tab = tab[(tab[target] > 0)] 
print('number of datapoints after drop = {}'.format(tab.shape[0]))

## Transform categorical data into a numerical representation
Categorical variables need to be modified to become numerical values. We can do this using pandas and sklearn have different tools to accomplish this.  We will use `pd.get_dummies` to perform one hot encoding - a technique used to convert categorical featurs into a binary matrix of 1s and 0s.  Below is a simple illustration of what this will do:

![](https://miro.medium.com/v2/resize:fit:1400/1*ggtP4a5YaRx6l09KQaYOnw.png)

Now let's apply this function to our data and inspect `tab`.

In [ ]:
tab['field'].value_counts()

In [ ]:
tab = pd.get_dummies(tab, columns=['field'],dtype=float)
tab

## Defining Features
Next, we will define the features. 

Let's drop a few of the features. The 'Av', 'lmass' and 'z_peak' values were all computed via FAST photometric fit, and so we will exclude them as well. In addition, we will exclude the categorical flag variables ('flags', 'f140w_flag', 'star_flag', 'use_phot', 'near_star' and 'field').

In [ ]:
features = [col for col in tab.columns if (col != target)]
features = [col for col in features if (col != 'Av') and (col != 'lmass') and (col != 'z_peak') 
            and (col != 'flags') and (col != 'f140w_flag') and (col != 'star_flag') 
            and (col != 'use_phot') and (col != 'near_star') and (col != 'field') 
            and (col != 'field_UDS') and (col != 'ra') and (col != 'dec')]

In [ ]:
features

## Check for and Imputing missing values 

In this particular dataset, we know via documentation that missing values of photometric errors are set to -99.  

Let's assume that we didn't have that info, What's some sanity checks we could do with our data to look for issues?  

In [ ]:
# We shouldn't have negative errors -- check for negative values
tab[tab['e_F606W'] < 0].e_F606W.value_counts()

### Imputing missing values

There are several options for what to do with missing values in our datasets.  Here are a few options:

1. Impute missing values with mean, median, or another ML model like KNNs. 
2. I could drop that row from dataset 
3. Depending on the model I am using, I could leave as -99.  (This will not work with linear regression, but could work with ensemble methods we will cover later)

We will impute missing values of photometric errors (set to -99. in the table) by assigning them the median of the distribution.

In [ ]:
# get columns associated with errors
columns = [col for col in features if ((col[:1] == 'e')or(col[:1] == 'f')) and (col[-1:] == 'W')]
columns

In [ ]:
# impute values with median
for column in columns:
    tab[column] = np.where(tab[column] < -90, tab[column].median(), tab[column])

Finally lets inspect distribution of features

In [ ]:
fig = plt.figure(0, [20, 18])

for k, feat in enumerate(features):
    ax = fig.add_subplot(7, 6, k+1)
    ax.hist(tab[feat], bins=50, log=True, color='navy')
    ax.set_title(feat)

ax = fig.add_subplot(7, 6, len(features)+1)
ax.hist(tab[target], bins=50, color='red')
ax.set_title(r'$z_{\rm spec}$')

plt.tight_layout()
plt.show()

## Build and Evaluate Baseline Models 

When building machine learning models its good to start with a baseline model.  This will give you a place to reference back to in terms of the evaluation metrics you are computing. In this notebook, we will start with a few simple things just to get a feel of what performance we can expect from our models:

1. **Mean Regressor:** we predict our target is the mean of all the target values contained in our dataset
2. Multiple Linear regression with no feature engineering


### Test / Train Split 

Let's get started by splitting the data we have into three groups:

1. Training: Data used to train the model
2. Testing: Data used to evaluate models performance when performing model tuning
3. Validation: Data used to evaluate the models performance on unseen data points 

In [ ]:
# Define X (features ) and y (target) for the dataset we are using
X = tab[features]
y = tab[target]

# first reserve 70% of the data for training, 30% for validation
X_train, X_validate, y_train, y_validate= train_test_split(X, y,  
                                                           test_size=0.3, 
                                                           random_state=42)

# second, split the validation set in half to obtain validation and test sets. 
X_validate, X_test, y_validate, y_test = train_test_split(X_validate, 
                                                          y_validate,  
                                                          test_size=0.5, 
                                                          random_state=42)

### Mean Regressor 

Next, lets build our mean regressor.  This can be done using sklearn's `DummyRegressor` class. 

In [ ]:
from sklearn.dummy import DummyRegressor

dummy_regr = DummyRegressor(strategy="mean")
dummy_regr.fit(X_train, y_train)

dummy_regr.predict(X_test)[0:10]

In [ ]:
# evaluate model 

# get predictions
y_predict = dummy_regr.predict(X_test) 
# compute mse with sklearns built in function
mse_dummy_regr= mean_squared_error(y_test, y_predict )
print('Mean squared error for mean regressor baseline: {}'.format(mse_dummy_regr))

Great! Now we have a metric that we really should beat from this point on.

## Naive Multiple Linear Regression with no feature engineering / inspection

Another thing we can do is just try multiple linear regression with our data.  I wouldn't expect good results here, but by comparing the results of this with the mean regressor will give us a sense for if there is signal in our data as the features currently are for linear regression.  

When building linear regression models, we have a choice of whether we standardize our data.  There are pros to either decision:

1. **Standardize data:** All of our features are on the same scale which means we can compare our learned coefficients.  This could give us insight into which of our features are the most helpful for making predictions.  However, we can interpret slopes 
2. **Do not standardize data:** We maintain the interpretation of coefficients.

I would like to use this initial pass to gain some instinct for what features are most important, so I will standardize our data and then compare the coefficients using `StandardScaler`.  `StandardScaler` transforms columns of your data to have mean 0 and standard deviation 1. 

![](https://journaldev.nyc3.cdn.digitaloceanspaces.com/2020/10/Standardization.png)

In [ ]:
from sklearn.linear_model import LinearRegression

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
# computes mean (mu) and standard deviation (sigma) with training
scaler.fit(X_train)  

# scales data with mu and sigma computed with .fit()
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

Finally lets build and evaluate our model.

In [ ]:
# train multiple linear regression model
linreg = LinearRegression().fit(X_train_scaled, y_train)

# evaluate model 
y_predict = linreg.predict(X_test_scaled) # get predictions
mse_mlr=mean_squared_error(y_test, y_predict )
print('Mean squared error for multiple linear regression baseline: {}'.format(mse_mlr))

Since we standardized our data we can compare the coefficients to see what feature is playing the biggest role in making predictions. Below I place the coefficients in to a dataframe.

In [ ]:
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': linreg.coef_, 'importance':np.absolute(linreg.coef_)})
coef_df.sort_values(by='importance',ascending=False)

Now, lets take the information above and see if we can use it to dive deeper into a few features

## Simple Linear Regression with Feature Engineering

Let's explore some of our top performing feature based on analysis above: f_F125W. First let's just see what the relationship looks like between feature and target.  

**Exercise**: Try uncommenting log scales and see what you discover.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(tab.f_F125W,tab.z_spec,alpha=0.1)
ax.set_xlabel('f_F125W')
ax.set_ylabel('z spec')

################################
# Uncomment lines to see how plot changes 
################################
#ax.set_yscale('log')
#ax.set_xscale('log')

Now lets build a a few models with simple linear regression where we engineer our feature and target; in this example we take the log of both the feature and target


In [ ]:
simplelinreg_log = LinearRegression().fit(np.log(X_train[['f_F125W']]), np.log(y_train))

Next lets evaluate our model.  It is important to note that if we are predicting the log(z_spec) with our linear regression model, we will need to convert the models prediction back to z_spec by using `np.exp()`. 

In [ ]:
# evaluate linreg_log model 
y_predict = np.exp(simplelinreg_log.predict(np.log(X_test[['f_F125W']]))) # get note, we undo the log of the target by using np.exp
mse_linreg_log = mean_squared_error(y_test, y_predict )

# compare performance of our simple linear regression models with our baselines
print('Mean squared error for mean regressor baseline: {}'.format(mse_dummy_regr))
print('Mean squared error for multiple linear regression baseline: {}'.format(mse_mlr))
print('The MSE for linreg is {}'.format(mse_linreg_log))

OK, nice! By using simple linear regression with engineering features, we now have a better score than our initial pass with MLR.  

Next, we will visualize our models prediction against the data below.

In [ ]:
fig,ax = plt.subplots(2)

logx = np.log(tab[['f_F125W']])
logy= np.log(tab.z_spec)
ax[0].scatter(logx, logy, marker= 'o', s=50, alpha=0.2, label='training data')
ax[0].plot(logx, simplelinreg_log.coef_ * logx+ simplelinreg_log.intercept_, 'r-', label='model')
#ax[0].set_title('Least-squares linear regression')
ax[0].set_xlabel('Log( Feature value (x) )')
ax[0].set_ylabel('Log( Target value (y))')
ax[0].legend()

# values for drawing line
x = np.linspace(-2,10,100)
y_predictions = np.exp( simplelinreg_log.coef_ * x+ simplelinreg_log.intercept_ )

ax[1].scatter(tab.f_F125W, tab.z_spec, marker= 'o', s=50, alpha=0.2, label='training data')
ax[1].plot(np.exp(x), y_predictions, 'r-', label='model')
#ax[1].set_title('Least-squares linear regression')
ax[1].set_xlabel('Feature value (x)')
ax[1].set_ylabel('Target value (y)')
ax[1].legend()

fig.tight_layout()

## OPTIONAL: Explore Regularization

Finally, let's see if we use some regularization techniques if that improves performance of our model.  When using a regularization technique it is important to explore the hyper paramter $\alpha$.  In the code below we will build several models for both Ridge and Lasso regresion using different values of $\alpha$.

In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn import preprocessing

alphas = np.logspace(-7, 7, 11)
models_lasso={}
models_ridge={}
for alpha in alphas:
    # build model
    model = Lasso(alpha=alpha, max_iter=100000)
    model.fit(X_train_scaled, y_train)
    models_lasso[alpha]=model

    # build model
    modelr = Ridge(alpha=alpha, max_iter=1000000)
    modelr.fit(X_train_scaled, y_train)
    models_ridge[alpha]=modelr

Next, we will evaluate these models with our testing data and plot how our performance changes with $\alpha$.

In [ ]:
# function to evaluate models 
def evaluate(model,X_test_scaled,y_test):
    y_predict = np.exp(model.predict(X_test_scaled)) # get predictions and undo log of target
    return mean_squared_error(np.exp(y_test), y_predict ) # compute MSE

# compute MSE for all values of alpha
mse_array_lasso = [mean_squared_error(model.predict(X_test_scaled),y_test) for model in models_lasso.values()]
mse_array_ridge = [mean_squared_error(model.predict(X_test_scaled),y_test) for model in models_ridge.values()]

In [ ]:
# plot results and print performance of best model
fig,ax=plt.subplots()
ax.plot(alphas,mse_array_lasso,label='lasso')
ax.scatter(alphas,mse_array_lasso)

ax.plot(alphas,mse_array_ridge,label='ridge')
ax.scatter(alphas,mse_array_ridge)

ax.scatter(alphas[np.argmin(mse_array_lasso)],np.array(mse_array_lasso).min(), c='k', s=200,alpha=0.5,label='Best score lasso')
ax.scatter(alphas[np.argmin(mse_array_ridge)],np.array(mse_array_ridge).min(), c='r', s=100,alpha=0.5,label='Best score ridge')

ax.legend()
ax.set_xscale('log')
ax.set_ylabel('Mean Squared Error')
ax.set_xlabel("$\\alpha$")

print("Best score from Ridge Regression {}".format(mse_array_ridge[np.argmin(mse_array_ridge)]))
print("Best score from Lasso Regression {}".format(mse_array_lasso[np.argmin(mse_array_lasso)]))

Something interesting we can do with regularization is see how are linear regression coefficients change with alpha. Below we plot this for both Ridge and Lasso regression. 

In [ ]:
def plot_alpha_v_coef(ax, alphas, coefs, column_names, method='Lasso'):
    '''
    plots alpha versus the beta coefficients 
    '''
    for feature in range(len(coefs[0])):
        if np.absolute(coefs[0, feature]) > 0.15:
            ax.plot(alphas, coefs[:, feature],
                     label="$\\beta_{{{}}}$".format(column_names[feature]))  #'{} order'.format(feature+1))
    ax.set_xscale('log')
    ax.set_title("$\\beta$ as a function of $\\alpha$ for {} regression".format(method))
    ax.set_xlabel("$\\alpha$")
    ax.set_ylabel("$\\beta$")
    ax.legend(loc="upper left",bbox_to_anchor=(1,1))

# get coef matrix
coefs_lasso = np.zeros((len(alphas), len(X.columns)))
for i,model in enumerate(models_lasso.values()):
    coefs_lasso[i] = model.coef_
coefs_ridge = np.zeros((len(alphas), len(X.columns)))
for i,model in enumerate(models_ridge.values()):
    coefs_ridge[i] = model.coef_
    
fig,ax=plt.subplots(2,figsize=(6,8))
plot_alpha_v_coef(ax[0], alphas, coefs_lasso, X.columns, method='Lasso')
plot_alpha_v_coef(ax[1], alphas, coefs_ridge, X.columns, method='Ridge')
fig.tight_layout()

## Print Performance of All Models 

In [ ]:
print('Mean squared error for mean regressor baseline: {}'.format(mse_dummy_regr))
print('Mean squared error for multiple linear regression baseline: {}'.format(mse_mlr))
print('The MSE for linreg is {}'.format(mse_linreg_log))
print("Best score from Ridge Regression {}".format(mse_array_ridge[np.argmin(mse_array_ridge)]))

In [ ]:
# compare error to z_peak and see how far we have to go 
print("Best score from Astronomy Paper {}".format(mean_squared_error(tab.loc[X_test.index].z_spec,tab.loc[X_test.index].z_peak)))

## Exercise:

Can you do better than what we have done so far by engineering new features? 

Ideas:
- take our highest performing simple linear regression model and try adding and additional feature(s) to see if you can get additional performance gains. 
- What happens to performance if we log our targets and all flux measurements?
    * Beware of collinear features 
- Is there a better simple linear regression model than the one we built? 